1. Importing Libraries

In [ ]:
import numpy as np 
import matplotlib.pyplot as plt
from glob import glob
import cv2
import pandas as pd

from keras.models import Model, Sequential
from keras.layers import Dense, Flatten, Conv2D, MaxPooling2D

import os
import seaborn as sns
from keras.applications.densenet import DenseNet169,DenseNet121,DenseNet201
from keras.applications.nasnet import NASNetMobile,NASNetLarge
from keras.applications.xception import Xception
from keras.applications.resnet import ResNet101, ResNet50, ResNet152
from keras.applications.resnet_v2 import ResNet50V2,ResNet101V2,ResNet152V2
from keras.applications.mobilenet_v2 import MobileNetV2
from keras.applications.mobilenet import MobileNet
from keras.applications.inception_resnet_v2 import InceptionResNetV2
from keras.applications.inception_v3 import InceptionV3
from keras.applications.vgg19 import VGG19
from keras.applications.vgg16 import VGG16
from keras.applications.efficientnet import EfficientNetB0
from google.colab import drive

2. Mounting Google Drive

In [ ]:
drive.mount('/content/drive')

In [ ]:
SIZE = [224,224]
train_images = []
train_labels  = []

3. Dataset Loading and Class Definition

In [ ]:
images_path = {}
images_path["Covid"] = glob("/content/drive/MyDrive/Dataset/COVID-19-master/X-Ray Image DataSet/Covid-19/*.png")
images_path["Covid"] += glob("/content/drive/MyDrive/Dataset/COVID-19-master/X-Ray Image DataSet/Covid-19/*.jpeg")
images_path["Covid"] += glob("/content/drive/MyDrive/Dataset/COVID-19-master/X-Ray Image DataSet/Covid-19/*.jpg")

images_path["NoFindings"] = glob("/content/drive/MyDrive/Dataset/COVID-19-master/X-Ray Image DataSet/No_findings/*.png")
images_path["NoFindings"] += glob("/content/drive/MyDrive/Dataset/COVID-19-master/X-Ray Image DataSet/No_findings/*.jpeg")
images_path["NoFindings"] += glob("/content/drive/MyDrive/Dataset/COVID-19-master/X-Ray Image DataSet/No_findings/*.jpg")

images_path["Pneumonia"] = glob("/content/drive/MyDrive/Dataset/COVID-19-master/X-Ray Image DataSet/Pneumonia/*.png")
images_path["Pneumonia"] += glob("/content/drive/MyDrive/Dataset/COVID-19-master/X-Ray Image DataSet/Pneumonia/*.jpeg")
images_path["Pneumonia"] += glob("/content/drive/MyDrive/Dataset/COVID-19-master/X-Ray Image DataSet/Pneumonia/*.jpg")

print(images_path["Covid"])
print(images_path["NoFindings"])
print(images_path["Pneumonia"])

images_class = {
    "Covid": 0,
    "Pneumonia": 1,
    "NoFindings": 2
}

4. Image Preprocessing and Label Assignment

In [ ]:
for label in images_path:
    for image_path in images_path[label]:
        image = cv2.imread(image_path)
        image = cv2.resize(image,(224, 224))
        train_images.append(image)
        train_labels.append(images_class[label])

In [ ]:
images = np.array(train_images)
labels = np.array(train_labels)

5. Label Encoding

In [ ]:
from sklearn.preprocessing import LabelEncoder
label = LabelEncoder()
labels = label.fit_transform(labels)
labels.shape

In [ ]:
images = images/255.0

6. Pre-trained CNN Architecture Selection

In [ ]:
SIZE = 224
model = DenseNet169(include_top=False, input_shape=(224, 224, 3), pooling='avg')
#model =DenseNet121(include_top=False, input_shape=(SIZE, SIZE, 3), pooling="avg")
#model = DenseNet201(include_top=False, input_shape=(SIZE, SIZE, 3), pooling='avg')
#model = VGG16(include_top=False, input_shape=(SIZE, SIZE, 3), pooling='avg')
#model = VGG19(include_top=False, input_shape=(SIZE, SIZE, 3), pooling='avg')
#model = InceptionV3(include_top=False, input_shape=(SIZE, SIZE, 3), pooling='avg')
#model = MobileNet(include_top=False, input_shape=(SIZE, SIZE, 3), pooling='avg')
#model = MobileNetV2(include_top=False, input_shape=(SIZE, SIZE, 3), pooling='avg')
#model = ResNet101(include_top=False, input_shape=(SIZE, SIZE, 3), pooling='avg')
#model = ResNet50(include_top=False, input_shape=(SIZE, SIZE, 3), pooling='avg')
#model = ResNet50V2(include_top=False, input_shape=(SIZE, SIZE, 3), pooling='avg')
#model = ResNet101V2(include_top=False, input_shape=(SIZE, SIZE, 3), pooling='avg')
#model = ResNet152V2(include_top=False, input_shape=(SIZE, SIZE, 3), pooling='avg')
#model = InceptionResNetV2(include_top=False, input_shape=(SIZE, SIZE, 3), pooling='avg')
#model = ResNet152(include_top=False, input_shape=(SIZE, SIZE, 3), pooling='avg')
#model = Xception(include_top=False, input_shape=(SIZE, SIZE, 3), pooling='avg')
#model = NASNetMobile(include_top=False, input_shape=(SIZE, SIZE, 3), pooling='avg')
#model = EfficientNetB0(include_top=False, input_shape=(SIZE, SIZE, 3), pooling='avg')

6.1. Freezing the Pre-trained Model Layers

In [ ]:
for layer in model.layers:
    layer.trainable = False

6.2. Deep Feature Extraction

In [ ]:
features = model.predict(images)
features = features.reshape(features.shape[0],-1)

7. Feature Selection Using Chi-Square

In [ ]:
from sklearn.feature_selection import f_classif
from sklearn.feature_selection import mutual_info_classif
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_classif,chi2

def select_features(X_train, y_train, X_test,i):
    # configure to select all features
    fs = SelectKBest(score_func=chi2, k=i)
    #fs = SelectKBest(score_func=f_classif, k=i)
    # learn relationship from training data
    fs.fit(X_train, y_train)
    # transform train input data
    X_train_fs = fs.transform(X_train)
    # transform test input data
    X_test_fs = fs.transform(X_test)
    return X_train_fs, X_test_fs

8. Three-Class Classification Using XGBoost

In [ ]:
from sklearn.metrics import f1_score
import statistics
from sklearn.metrics import confusion_matrix,accuracy_score
from xgboost import XGBClassifier
from sklearn.metrics import precision_score
from sklearn.model_selection import train_test_split


x_train,x_test,y_train,y_test = train_test_split(features,labels,test_size=0.2)
x_train,x_test = select_features(x_train,y_train,x_test,1000)
classifier = XGBClassifier(learning_rate=0.50,n_estimators=100,seed=0)
classifier.fit(x_train,y_train)
yhat = classifier.predict(x_test)
acc = accuracy_score(y_test, yhat)
print(confusion_matrix(y_test, yhat))
Y_test = y_test
Y_pred = yhat
l = confusion_matrix(Y_test, Y_pred)

print(f"\nAccuracy score : ", accuracy_score(Y_test, Y_pred))
print(f"\nconfusion matrix : \n", confusion_matrix(Y_test, Y_pred))
print(f"\nf1 score : ", f1_score(Y_test, Y_pred, average="macro"))
print(f"\nprecision score : ", precision_score(Y_test, Y_pred, average="macro"))
print(f"\nSpecifity : ", (l[1,1]+l[2,2]) / (l[1,1]+l[2,2]+l[1,0]+l[2,0]))
print(f"\nSensitivity : ", l[0,0] / (l[0,0] + l[0,1]+l[0,2]))
print(f"\n============")

9 Confusion Matrix Visualization

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay
import seaborn as sn
import pandas as pd
import matplotlib.pyplot as plt

class_names=['Covid-19','No_findings','Pneumonia']
group_count = ['{0:0.0f}'.format(value) for value in
                l.flatten()]
group_counts=[]
group_percentages = []
for i in group_count:
    group_counts.append(f'({i})')
for i in range(0,3):
    for j in range(0,3):
        group_percentages.append(l[i,j]/np.sum(l[:,j]))
        
group_percentages = ['{0:.2%}'.format(value) for value in
            group_percentages]

labels = [f'{v2}\n{v3}' for  v2, v3 in
          zip(group_percentages,group_counts)]                     

labels = np.asarray(labels).reshape(3,3)                    

df_cm = pd.DataFrame(l, index = [i for i in class_names],
                  columns = [i for i in class_names])
plt.figure(figsize = (6,4))
ax = plt.axes()
sns = sn.heatmap(df_cm, annot=labels,ax = ax,cbar=False,fmt="",cmap="Blues")
ax.set_title('Overlapped',fontsize = 17)
plt.show()
figure = sns.get_figure()    
figure.savefig(f'Overlapped.svg')
plt.clf()  
